In [2]:
import os

In [4]:
# os.chdir("..")

In [5]:
%pwd

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model'

In [17]:
import requests
import pandas as pd
import sys
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from src.customer_churn.exception.exception import CustomerChurnException
pio.renderers.default = 'notebook'
%matplotlib inline

In [7]:
API_URL = "http://127.0.0.1:8000/"

# A dummy payload for testing:

In [76]:
payload = {  
            'order_frequency': 25,
            'total_monetary_value': -850,
            'total_quantity_abs': 10,
            'total_order_issues': 8,
            'country': 'Germany',
            'avg_order_value': -850,
            'avg_quantity_per_order': 250,
            'customer_order_issue_rate': 13,
            'recency': 25,
            'tenure': 97,
            'avg_stockcode_issue_rate' : 1,
            'max_stockcode_issue_rate': 1.02,
            'total_orders_made_for_stock': 195
            }

# Calling the predict endpoint:

In [77]:
response = requests.post(f"{API_URL}/predict_single_feature", json=payload)
results = response.json()

# Creating the `SHAP` visualization class:

In [78]:
class ShapVisualization:
    
    def __init__(self, results:dict):
        self.results = results

    
    def df_from_predict_api_response(self) -> pd.DataFrame:
        """
        Takes in the response of the predict endpoint, 
        converts it to a pandas dataframe with an extra column 'color' for visualization purpose
        """
        try:
            contributions = self.results['shap_values']
            sorted_contrib = sorted(contributions, key=lambda x: abs(x['shap_value']), reverse=True)
            features = [c['feature'] for c in sorted_contrib]
            values = [c['shap_value'] for c in sorted_contrib]
            input_features = [c['value'] for c in sorted_contrib]
            shap_df= pd.DataFrame({'Features':features, 'SHAP_values': values, 'Input_features':input_features})
            shap_df['Color'] = ['Decreases churn risk ↓' if v < 0 else 'Increases churn risk ↑' for v in shap_df['SHAP_values']]
            shap_df = shap_df.sort_values('SHAP_values', key=abs, ascending=False)
            return shap_df
        except Exception as e:
            raise CustomerChurnException(e, sys)


    def plot_shap_feature_importance(self):
        """
        Takes in the SHAP df and visualizes the bar graph based on the SHAP values to show the differences,
        where negative(-) SHAP value decreases the churn risk and positive(+) SHAP value increases the churn risk:
        Parameters:
        ----------
        df: a pandas dataframe consisting of the feature names, SHAP values, input features, and color column for the bar chart
        """
        try:
            shap_df = self.df_from_predict_api_response()
            net_shap = sum(shap_val for shap_val in shap_df['SHAP_values'])
            fig = px.bar(shap_df, 
                     x='Features', 
                     y='SHAP_values', 
                     custom_data= ['Input_features', 'Color'],
                     color='Color', 
                     width=1200,
                     height=950,
                     color_discrete_map={
                        'Increases churn risk ↑': '#b11346',
                        'Decreases churn risk ↓': '#0e7337'
                },
                title='SHAP Feature Importance',
                labels={'Color': 'Effect on Churn Risk',
                       'SHAP_values': 'SHAP Value (Impact on Churn Probability)',
                       'Features': 'Customer Feature'})
            fig.update_traces(
                hovertemplate = "<b>Impact:</b> %{customdata[1]}<br>" +
                                "<b>Feature value:</b> %{customdata[0]}<br>"+
                                "<b>SHAP value:</b> %{x}<br>"+
                                "<extra></extra>"
            )
            return fig, net_shap, shap_df
        except Exception as e:
            raise CustomerChurnException(e, sys)
    

In [79]:
shap_visualization = ShapVisualization(results=results)
fig, net_shap, shap_df = shap_visualization.plot_shap_feature_importance()

In [80]:
shap_df.columns

Index(['Features', 'SHAP_values', 'Input_features', 'Color'], dtype='object')

In [82]:
fig

'No Risk'

In [88]:
results['prediction_results']

[{'prediction_probability': 0.4312,
  'retention_probability': 0.5688,
  'prediction': 'Stay',
  'risk_label': 'No Risk'}]

In [11]:
# net_shap = sum(c['shap_value'] for c in contributions)
# if net_shap < 0:
#     st.caption(f"Net SHAP effect: {net_shap:.3f} — "
#                f"retention signals outweigh churn signals")
# else:
#     st.caption(f"Net SHAP effect: {net_shap:.3f} — "
#                f"churn signals outweigh retention signals")

In [13]:
# net_shap = sum(c['shap_value'] for c in contributions)
# if net_shap < 0:
#     print(f"Net SHAP effect: {net_shap:.3f} — "
#                f"retention signals outweigh churn signals")
# else:
#     print(f"Net SHAP effect: {net_shap:.3f} — "
#                f"churn signals outweigh retention signals")

In [75]:
results['prediction_results']

[{'prediction_probability': 0.4312,
  'prediction': 'Stay',
  'risk_label': 'No Risk'}]

In [ ]:
m1.metric("🟢 Retention" if results['prediction_results'][0]['prediction'] == "Stay" else "🔴 Churn")

In [ ]:
 m1, m2, m3 = st.columns(3)

    # Delta percentage:
    m1.metric("Churn Risk", 
            "🟢 Low" if prediction == 0 else "🔴 High",
            delta=f"{delta_precentage:.2f}% better than average" if prediction == 0 else f"{delta_precentage:.2f}% worse than average")
    
    # Prediction Probability:
    m2.metric(label= "Prediction Confidence", value=f"{prediction_prob:.2f}%", delta="Model's confidence")